# 04 Batch Predict

Predicts every item from an SQL pull (default: uncategorized items) or a file, using a saved model
and **that model's own cleaning recipe**. Writes `outputs/predictions_<timestamp>/`:

| file | contents |
|---|---|
| `predictions_<ts>.csv` | one row per input row, in the ensemble's format (comma-separated) |
| `metadata_<ts>.json` | model folder, hyperparameters, cleaning configuration, data source, row counts |
| `evaluation/` | only when the input carries true labels: accuracy by level and by confidence bucket |

Confidences are raw model scores (no calibration).

In [0]:
# ---- MLflow -------------------------------------------------------------------
USE_MLFLOW = False                    # False = no tracking at all
MLFLOW_EXPERIMENT = None              # None = this notebook's experiment; or e.g. "/Users/patrick.white@vallen.com/nb-classifier"

# ---- model --------------------------------------------------------------------
MODEL = None                          # None = newest folder in models/; or "model_20261016_120000" / "20261016_120000"

# ---- data source (RAW_DATA_FILE supersedes SQL_QUERY) -----------------------
RAW_DATA_FILE = None                  # e.g. "data/my_extract.txt"; read with dtype=str
RAW_DATA_DELIMITER = "|"              # delimiter of RAW_DATA_FILE
RAW_DATA_QUOTED = True                # False if RAW_DATA_FILE uses no quoting and descriptions may start with "
SQL_QUERY = "unlabeled_items"          # a file in data/queries/ (name without .sql); may be None only if RAW_DATA_FILE is set

# ---- items to exclude ---------------------------------------------------------
BLACKLIST_FILE = None                 # e.g. "data/blacklists/blacklist.txt" (one column)
BLACKLIST_COL = "Vallen ID"           # data column the blacklist values are matched against
BLACKLIST_HAS_HEADER = True           # first line of the blacklist file is a header

# ---- columns ------------------------------------------------------------------
TEXT_COL = None                       # None = the description column the model was trained on (e.g. "FullDesc")
KEEP_COLS = ["ID", "Vallen ID"]       # identifiers / extra columns copied into the output ("ID" = ensemble join key)
INCLUDE_TEXT = True                   # also write the raw and cleaned descriptions

# ---- evaluation (only used when the data contains the model's label columns) ----
BUCKET_WIDTH = 0.05                   # confidence bucket width; change and re-run the evaluation cells

In [0]:
# ---- setup (no settings below this line need changing) ----------------------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from nbclassifier import artifacts, data, evaluation, paths, predict
from nbclassifier.tracking import Tracker

TS = paths.timestamp()
tracker = Tracker(USE_MLFLOW, MLFLOW_EXPERIMENT, run_name=f"04_batch_predict_{TS}",
                  tags={"notebook": "04_batch_predict"})

## Load model

In [0]:
model = artifacts.load_model(MODEL)
clf, cleaner, man = model.clf, model.cleaner, model.manifest
LEVEL_COLUMNS = clf.level_columns
NAME_COLUMNS = [c for c in (model.lookup.columns if model.lookup is not None else []) if c not in LEVEL_COLUMNS]
TEXT_COL = TEXT_COL or model.training_metadata.get("text_col", "FullDesc")
CLEANED_COL = model.training_metadata.get("cleaned_col", "CleanDesc")

print(f"model     : {model.folder.name}  (created {man['created_utc']})")
print(f"trained on: {man['n_training_rows']:,} rows, {man['n_features']:,} features, {man['n_paths']:,} paths")
print(f"cleaning  : {cleaner}")
print("hyperparameters:")
for k, v in clf.hp.to_dict().items():
    print(f"    {k:<22}{v}")

model     : model_20261019_184618  (created 2026-10-19T18:48:10+00:00)
trained on: 808,884 rows, 229,579 features, 3,644 paths
cleaning  : Cleaner(method='cleaning_method_20261015', english_words=234,377, vocabulary=33,146 words, append_fields={})
hyperparameters:
    vectorizer_kind       tfidf
    max_features          None
    ngram_range           [1, 2]
    min_df                2
    max_df                0.95
    stop_words            english
    sublinear_tf          True
    binary                True
    norm                  l2
    use_idf               False
    ordered_bigrams       True
    token_pattern         whitespace
    alpha                 0.001
    fit_prior             True
    prediction_mode       conditional
    rare_path_threshold   5


## Load data

In [0]:
raw, source = data.load_raw(RAW_DATA_FILE, SQL_QUERY, delimiter=RAW_DATA_DELIMITER, quoted=RAW_DATA_QUOTED,
                            spark=globals().get("spark"))
n_raw = len(raw)
n_black = 0
if BLACKLIST_FILE is not None:
    items = data.read_blacklist(BLACKLIST_FILE, has_header=BLACKLIST_HAS_HEADER)
    raw, n_black = data.apply_blacklist(raw, items, BLACKLIST_COL)
n_dup = data.duplicate_count(raw, "ID")
print(f"raw rows: {n_raw:,} | blacklisted: {n_black:,} | to predict: {len(raw):,}")
if n_dup:
    print(f"WARNING: {n_dup:,} rows share an 'ID' with another row (the ensemble joins on ID)")

raw rows: 1,037,871 | blacklisted: 0 | to predict: 1,037,871


## Clean (with the model's recipe; every row is kept)

In [0]:
df, counts = data.clean_dataframe(raw, cleaner, TEXT_COL, CLEANED_COL, drop_empty=False)
n_empty = int((df[CLEANED_COL].str.strip() == "").sum())
print(f"rows: {len(df):,} | cleaned to nothing: {n_empty:,} (predicted from priors only; oov_rate = 1)")
stats = data.ngram_stats(df[CLEANED_COL])
data.print_summary({"raw_rows": n_raw, "total_dropped": n_raw - len(df), "cleaned_rows": len(df)}, stats)

rows: 1,037,871 | cleaned to nothing: 10,723 (predicted from priors only; oov_rate = 1)
raw data rows      :    1,037,871
rows dropped       :            0
cleaned data rows  :    1,037,871
unique tokens      :       34,509
unique bigrams     :      656,982
tokens + bigrams   :      691,491


## Predict

In [0]:
has_labels = all(c in df.columns for c in LEVEL_COLUMNS)
pred = predict.prediction_table(
    clf, df, CLEANED_COL,
    id_cols=KEEP_COLS,
    actual_cols=LEVEL_COLUMNS if has_labels else (),
    text_cols=[TEXT_COL, CLEANED_COL] if INCLUDE_TEXT else (),
)
pred = predict.attach_names(pred, model.lookup, LEVEL_COLUMNS, NAME_COLUMNS)
print(f"{len(pred):,} predictions; labels present: {has_labels}")
pred.head()

1,037,871 predictions; labels present: False


,ID,Vallen ID,Predicted Level 1 ID,Predicted Level 2 ID,Predicted Level 3 ID,Predicted Level 4 ID,Predicted Level 1 Name,Predicted Level 2 Name,Predicted Level 3 Name,Predicted Level 4 Name,L1 Confidence,L2 Confidence,L3 Confidence,L4 Confidence,L1 Margin,L2 Margin,L3 Margin,L4 Margin,n_tokens,n_known,oov_rate,FullDesc,CleanDesc
0,2380371,FRX I400,L1-12798733,L2-12798746,L3-12798786,L4-12799046,Electrical & Lighting,Industrial & Commercial Lighting,Lamps & Bulbs,Incandescent Lamps,0.227507,0.142823,0.115383,0.110038,-0.181773,-0.117448,-0.088827,0.013972,3,3,0.0,INSPECTION WATER TYPE,INSPECTION WATER TYPE
1,2380873,PHF LRF101,L1-12810533,L2-12810534,L3-12810563,L4-12810738,Tools & Equipment,Hand Tools,Sockets & Drive Tools,Socket Sets,0.745428,0.670176,0.436604,0.169325,0.564728,0.489864,0.244427,0.047473,4,4,0.0,RANGE FINDER 1000 VORTEX IMPACT,RANGE FINDER VORTEX IMPACT
2,2387614,VEN PCPLS4278,L1-12798733,L2-12798738,L3-12798764,L4-12798872,Electrical & Lighting,Control & Automation,Sensors & Measurement Devices,Sensors,0.659812,0.322648,0.250396,0.249679,0.446699,0.167199,0.146721,0.146004,2,2,0.0,VENDING SCANNER,VENDING SCANNER
3,404483F-900YLW.5OZ,404483F-900YLW.5OZ,L1-12800172,L2-12800187,L3-12800189,L4-12800200,"Paint, Coatings & Marking",Paint & Equipment,"Paint, Primers & Thinners",Paint Thinners,0.669266,0.396252,0.313883,0.223061,0.531232,0.123248,0.040879,0.116151,2,2,0.0,F900-YELLOW SEAL LACQUER,SEAL LACQUER
4,75416618459,75416618459,L1-12783695,L2-12783697,L3-12783719,L4-12784326,Cutting Tools,Holemaking,Masonry & Concrete Drill Bits,Masonry Bits,0.742426,0.742181,0.641443,0.602993,0.495042,0.525966,0.426743,0.495220,2,2,0.0,2662512 5/8 X 12 SDS,<SIZE> SDS


## Save predictions

In [0]:
OUT_DIR = paths.new_folder(paths.OUTPUTS_DIR, "predictions_", TS)
pred_path = OUT_DIR / f"predictions_{TS}.csv"
pred.to_csv(pred_path, index=False)
metadata = {
    "created": TS,
    "model_folder": model.folder.name,
    "model_manifest": man,
    "hyperparameters": clf.hp.to_dict(),
    "cleaning": cleaner.describe(),
    "source": source,
    "blacklist_file": str(paths.resolve(BLACKLIST_FILE)) if BLACKLIST_FILE else None,
    "blacklist_col": BLACKLIST_COL,
    "row_counts": {"raw_rows": n_raw, "blacklisted_dropped": n_black, "predicted_rows": len(pred),
                   "cleaned_to_nothing": n_empty},
    "ngram_stats": stats,
    "columns": list(pred.columns),
    "has_labels": has_labels,
}
data.write_json(OUT_DIR / f"metadata_{TS}.json", metadata)
print("wrote", pred_path)

tracker.log_params({"model_folder": model.folder.name, "predictions_folder": OUT_DIR.name,
                    "source": {k: v for k, v in source.items() if k != "query_text"}})
tracker.log_metrics({**metadata["row_counts"], **stats})
tracker.log_artifact(OUT_DIR / f"metadata_{TS}.json")

wrote /Workspace/Users/patrick.white@vallen.com/Category Classification/New NB Repo/outputs/predictions_20261019_190511/predictions_20261019_190511.csv


## Evaluate (only if the data contained true labels)

Headline accuracy by level, then accuracy by confidence bucket. Change `BUCKET_WIDTH` and re-run
from the next cell; nothing is re-predicted.

In [0]:
if has_labels:
    labelled = pred.dropna(subset=[f"Actual {c}" for c in LEVEL_COLUMNS])
    print(f"{len(labelled):,} labelled rows")
    head = evaluation.headline(labelled, LEVEL_COLUMNS)
    display(head)
    tracker.log_metrics(head.loc["all"].to_dict())
else:
    print("no label columns in the data; skipping evaluation")

no label columns in the data; skipping evaluation


In [0]:
# BUCKET_WIDTH = 0.1   # <- change here and re-run this cell and the next
if has_labels:
    EVAL_DIR = OUT_DIR / "evaluation"
    buckets = evaluation.bucket_tables(labelled, LEVEL_COLUMNS, width=BUCKET_WIDTH)
    EVAL_DIR.mkdir(exist_ok=True)
    buckets.to_csv(EVAL_DIR / f"buckets_width_{BUCKET_WIDTH}.csv", index=False)
    display(buckets[buckets["level"] == f"L{len(LEVEL_COLUMNS)}"])

In [0]:
if has_labels:
    fig = evaluation.plot_accuracy_by_bucket(labelled, LEVEL_COLUMNS, width=BUCKET_WIDTH, target_accuracy=0.95)
    evaluation.save_figure(fig, EVAL_DIR, f"accuracy_by_bucket_by_level_w{BUCKET_WIDTH}", tracker)
    fig = evaluation.plot_coverage(labelled, LEVEL_COLUMNS)
    evaluation.save_figure(fig, EVAL_DIR, "coverage_by_level", tracker)

In [0]:
tracker.end()